# CalCOFI Big Data Analytics Solution
## One-year-ahead station-level hypoxia warning

This Google Colab notebook implements the A5 BDAS workflow with **PySpark and Spark MLlib**. It predicts whether a regularly sampled CalCOFI station will have a low minimum dissolved-oxygen observation in the following survey year.

**Project scope:** California Current observations, years 2000 onward. The analysis uses station-year aggregates so that the prediction unit is a station in one predictor year and the outcome belongs to the same station in the following year.

Run cells from top to bottom. Download this notebook and the generated `output` folder when finished; these form the Colab-code/data-output submission evidence.

In [ ]:
# Colab includes a PySpark runtime. Do not install a different version because it can conflict with Colab's Dataproc packages.
import pyspark
print('Using PySpark version:', pyspark.__version__)


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

spark = (SparkSession.builder
         .appName('CalCOFI_BDAS')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
# The workflow runs on one Colab runtime; fewer shuffle partitions reduce task overhead without changing results.
spark.conf.set('spark.sql.shuffle.partitions', '64')

OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
print('Spark version:', spark.version)


## 1. Collect initial data

The two CalCOFI input archives are versioned with this notebook in the GitHub repository under `A5/datasets/`. This cell downloads them to the temporary Colab runtime and extracts the original CSV files. No manual upload is required.

In [ ]:
# Download the repository versions once per Colab runtime, then extract them for Spark.
from urllib.request import urlretrieve

REPOSITORY_DATA_URL = 'https://raw.githubusercontent.com/graham-gc/infosys722/main/A5/datasets'
# Change to True when the archives in GitHub have been updated during this Colab runtime.
REFRESH_DATA = False
DATA_DIR = Path('/content/datasets')
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVES = ['194903-202105_Bottle.csv.zip', '194903-202105_Cast.csv.zip']

for archive_name in ARCHIVES:
    archive_path = DATA_DIR / archive_name
    if REFRESH_DATA or not archive_path.exists():
        urlretrieve(f'{REPOSITORY_DATA_URL}/{archive_name}', archive_path)
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(DATA_DIR)

SEARCH_DIRS = [DATA_DIR]

def locate_file(candidates):
    for directory in SEARCH_DIRS:
        for name in candidates:
            path = directory / name
            if path.exists():
                return str(path)
    raise FileNotFoundError(f'Could not find any of: {candidates}. Upload the Bottle and Cast CSV files, then rerun this cell.')

BOTTLE_PATH = locate_file(['194903-202105_Bottle.csv'])
CAST_PATH = locate_file(['194903-202105_Cast.csv'])
print('Bottle:', BOTTLE_PATH)
print('Cast:', CAST_PATH)


In [ ]:
def read_calcofi_csv(path):
    return (spark.read
            .option('header', True)
            .option('inferSchema', True)
            .option('mode', 'PERMISSIVE')
            .csv(path))

bottle_raw = read_calcofi_csv(BOTTLE_PATH)
cast_raw = read_calcofi_csv(CAST_PATH)

collection_summary = pd.DataFrame([
    {'table': 'Bottle', 'source_file': Path(BOTTLE_PATH).name, 'rows': bottle_raw.count(), 'columns': len(bottle_raw.columns)},
    {'table': 'Cast', 'source_file': Path(CAST_PATH).name, 'rows': cast_raw.count(), 'columns': len(cast_raw.columns)},
])
display(collection_summary)
collection_summary.to_csv(OUTPUT_DIR / '01_collection_summary.csv', index=False)

print('Bottle schema')
bottle_raw.printSchema()
print('Cast schema')
cast_raw.printSchema()


## 2. Describe, explore and verify data quality

The project uses direct dissolved oxygen (`O2ml_L`) as the authoritative environmental measurement. Temperature, salinity and nutrients are predictor candidates. The quality report below makes missingness explicit before any rows are removed.

In [ ]:
bottle_fields = ['Cst_Cnt', 'Sta_ID', 'Depthm', 'T_degC', 'Salnty', 'O2ml_L', 'O2Sat', 'PO4uM', 'NO3uM', 'SiO3uM', 'ChlorA', 'RecInd']
cast_fields = ['Cst_Cnt', 'Year', 'Month', 'Lat_Dec', 'Lon_Dec', 'Bottom_D', 'Sta_Code', 'Data_Type']

# Calculate every field's missingness in one Spark aggregation, rather than scanning the Bottle table once per field.
quality_fields = [field for field in bottle_fields if field in bottle_raw.columns]
missing_counts = (bottle_raw.agg(
    F.count(F.lit(1)).alias('__total__'),
    *[F.sum(F.when(F.col(field).isNull(), 1).otherwise(0)).alias(field) for field in quality_fields]
).first().asDict())
total = missing_counts.pop('__total__')
missing_rows = [
    {'field': field, 'missing_rows': int(missing_counts[field]), 'completeness_pct': round(100 * (1 - missing_counts[field] / total), 3)}
    for field in quality_fields
]
quality_report = pd.DataFrame(missing_rows).sort_values('completeness_pct')
display(quality_report)
quality_report.to_csv(OUTPUT_DIR / '02_bottle_completeness.csv', index=False)

recind_distribution = (bottle_raw.groupBy('RecInd').count().orderBy('RecInd').toPandas()) if 'RecInd' in bottle_raw.columns else pd.DataFrame()
display(recind_distribution)
recind_distribution.to_csv(OUTPUT_DIR / '03_record_indicator_distribution.csv', index=False)


In [ ]:
# Raw-data visualisations: year coverage and dissolved-oxygen distribution before preparation.
cast_years = cast_raw.select('Cst_Cnt', 'Year').dropDuplicates(['Cst_Cnt'])
raw_oxygen = (bottle_raw.select('Cst_Cnt', F.col('O2ml_L').cast('double').alias('O2ml_L'))
    .join(F.broadcast(cast_years), 'Cst_Cnt', 'inner')
    .filter(F.col('Year') >= 2000)
    .persist(StorageLevel.MEMORY_AND_DISK))
raw_oxygen.count()
raw_sample = raw_oxygen.filter(F.col('O2ml_L').isNotNull()).sample(False, 0.02, seed=722).limit(100000).toPandas()
year_counts = raw_oxygen.groupBy('Year').count().orderBy('Year').toPandas()
year_counts.to_csv(OUTPUT_DIR / '04_raw_year_counts.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.lineplot(data=year_counts, x='Year', y='count', marker='o', ax=axes[0], color='#1f5a99')
axes[0].set_title('Raw oxygen observations by survey year')
axes[0].set_ylabel('Bottle observations')
sns.histplot(raw_sample['O2ml_L'], bins=60, ax=axes[1], color='#1f5a99')
axes[1].axvline(1.4, color='crimson', linestyle='--', label='Project warning threshold')
axes[1].set_title('Raw dissolved oxygen distribution, year ≥ 2000')
axes[1].set_xlabel('O2ml_L')
axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_raw_data_exploration.png', dpi=220, bbox_inches='tight')
plt.show()
raw_oxygen.unpersist()


In [ ]:
# Additional raw-data context for KDD Step 2.3: seasonality, spatial coverage and sampling depth.
raw_context = (bottle_raw.select(
        'Cst_Cnt', F.col('Depthm').cast('double').alias('Depthm'),
        F.col('O2ml_L').cast('double').alias('O2ml_L'))
    .join(F.broadcast(cast_raw.select(
        'Cst_Cnt', 'Year', 'Month', 'Lat_Dec', 'Lon_Dec').dropDuplicates(['Cst_Cnt'])), 'Cst_Cnt', 'inner')
    .filter(F.col('Year') >= 2000)
    .persist(StorageLevel.MEMORY_AND_DISK))
raw_context.count()
month_counts = (raw_context.filter(F.col('Month').between(1, 12))
    .groupBy('Month').count().orderBy('Month').toPandas())
month_counts.to_csv(OUTPUT_DIR / '04b_raw_month_counts.csv', index=False)
context_sample = (raw_context
    .filter(F.col('O2ml_L').isNotNull() & F.col('Depthm').isNotNull() &
            F.col('Lat_Dec').isNotNull() & F.col('Lon_Dec').isNotNull())
    .sample(False, 0.02, seed=722).limit(100000).toPandas())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
sns.barplot(data=month_counts, x='Month', y='count', ax=axes[0], color='#4c78a8')
axes[0].set(title='Raw observations by calendar month', xlabel='Month', ylabel='Bottle observations')
spatial = axes[1].scatter(context_sample['Lon_Dec'], context_sample['Lat_Dec'],
    c=context_sample['O2ml_L'], cmap='viridis', s=10, alpha=0.55)
fig.colorbar(spatial, ax=axes[1], label='Dissolved oxygen (mL/L)')
axes[1].set(title='Geographic coverage of seeded 2% sample', xlabel='Longitude', ylabel='Latitude')
sns.histplot(context_sample['Depthm'], bins=60, ax=axes[2], color='#59a14f')
axes[2].set_xscale('symlog', linthresh=1)
axes[2].set_xlim(left=0)
axes[2].set(title='Raw sampling-depth distribution', xlabel='Depth (m; symlog scale)', ylabel='Bottle observations')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04b_raw_sampling_context.png', dpi=240, bbox_inches='tight')
plt.show()
raw_context.unpersist()


## 3. Data preparation: select, clean, integrate, construct and reformat

The Cast table provides time and location; the Bottle table provides sampled water chemistry. They are integrated using `Cst_Cnt`. The primary analysis keeps observed bottle records (`RecInd = 3` where that field is present), years 2000 onward, plausible oxygen values (0–12 mL/L), and rows with a station identifier and coordinates. The aggregation is intentional vertical reduction: many bottle depths become one comparable station-year record.

The one-year-ahead outcome is the **minimum observed oxygen in the same station in the following calendar year**. `1.4 mL/L` is a project warning threshold, not a universal ecological mortality threshold.

In [ ]:
# Select required fields and make names unambiguous before the join.
bottle = bottle_raw.select(*[c for c in bottle_fields if c in bottle_raw.columns])
cast = cast_raw.select(*[c for c in cast_fields if c in cast_raw.columns])

# Audit Cast_Cnt before applying defensive deduplication.
cast_key_audit = (cast.agg(
    F.count(F.lit(1)).alias('cast_rows'),
    F.count('Cst_Cnt').alias('non_null_cast_keys'),
    F.countDistinct('Cst_Cnt').alias('distinct_cast_keys'),
)
.withColumn('null_cast_keys', F.col('cast_rows') - F.col('non_null_cast_keys'))
.withColumn('duplicate_key_excess', F.col('non_null_cast_keys') - F.col('distinct_cast_keys'))
.toPandas())
display(cast_key_audit)
cast_key_audit.to_csv(OUTPUT_DIR / '01b_cast_key_audit.csv', index=False)

# Cst_Cnt should identify one sampling event. Deduplication remains a defensive merge control.
cast = cast.dropDuplicates(['Cst_Cnt'])

joined = (bottle
    .join(F.broadcast(cast), on='Cst_Cnt', how='inner')
    .withColumn('Year', F.col('Year').cast('int'))
    .withColumn('O2ml_L', F.col('O2ml_L').cast('double'))
    .withColumn('T_degC', F.col('T_degC').cast('double'))
    .withColumn('Salnty', F.col('Salnty').cast('double'))
    .withColumn('Depthm', F.col('Depthm').cast('double'))
    .withColumn('PO4uM', F.col('PO4uM').cast('double'))
    .withColumn('NO3uM', F.col('NO3uM').cast('double'))
    .withColumn('SiO3uM', F.col('SiO3uM').cast('double'))
    .persist(StorageLevel.MEMORY_AND_DISK))

# Record the cumulative effect of each preparation rule in one Spark aggregation.
year_ok = F.col('Year') >= 2000
spatial_ok = F.col('Sta_ID').isNotNull() & F.col('Lat_Dec').isNotNull() & F.col('Lon_Dec').isNotNull()
oxygen_ok = F.col('O2ml_L').between(0, 12)
quality_ok = (F.col('RecInd').cast('int') == 3) if 'RecInd' in joined.columns else F.lit(True)
oxygen_quality_detail = (joined.filter(year_ok & spatial_ok).agg(
    F.count(F.lit(1)).alias('post_2000_spatial_rows'),
    F.sum(F.when(F.col('O2ml_L').isNull(), 1).otherwise(0)).alias('missing_oxygen_rows'),
    F.sum(F.when(F.col('O2ml_L') < 0, 1).otherwise(0)).alias('oxygen_below_zero_rows'),
    F.sum(F.when(F.col('O2ml_L') > 12, 1).otherwise(0)).alias('oxygen_above_12_rows'),
    F.sum(F.when(F.col('O2ml_L').between(0, 12), 1).otherwise(0)).alias('oxygen_in_range_rows'),
).toPandas())
display(oxygen_quality_detail)
oxygen_quality_detail.to_csv(OUTPUT_DIR / '14b_data_quality_detail.csv', index=False)
audit_stages = [
    ('Joined Bottle–Cast records', F.lit(True), 'Inner join on Cst_Cnt'),
    ('Year scope retained', year_ok, 'Year >= 2000'),
    ('Station and coordinates retained', year_ok & spatial_ok, 'Sta_ID, latitude and longitude present'),
    ('Plausible oxygen retained', year_ok & spatial_ok & oxygen_ok, '0 <= O2ml_L <= 12'),
]
if 'RecInd' in joined.columns:
    audit_stages.append(('Observed records retained', year_ok & spatial_ok & oxygen_ok & quality_ok, 'RecInd = 3'))
audit_values = joined.agg(*[
    F.sum(F.when(condition, 1).otherwise(0)).alias(f'stage_{index}')
    for index, (_, condition, _) in enumerate(audit_stages)
]).first().asDict()
audit_rows = []
previous_rows = None
for index, (stage, _, rule) in enumerate(audit_stages):
    remaining = int(audit_values[f'stage_{index}'])
    audit_rows.append({
        'stage': stage,
        'rule': rule,
        'records_remaining': remaining,
        'removed_from_previous_stage': 0 if previous_rows is None else previous_rows - remaining,
    })
    previous_rows = remaining
preparation_audit = pd.DataFrame(audit_rows)
display(preparation_audit)
preparation_audit.to_csv(OUTPUT_DIR / '14_data_preparation_audit.csv', index=False)

final_filter = year_ok & spatial_ok & oxygen_ok & quality_ok
prepared = joined.filter(final_filter).persist(StorageLevel.MEMORY_AND_DISK)

print('Prepared bottle records:', prepared.count())
print('Prepared Cast events:', prepared.select('Cst_Cnt').distinct().count())
joined.unpersist()


In [ ]:
# Construct station-year predictors. Mean and minimum oxygen represent different aspects of current conditions.
station_year = (prepared
    .groupBy('Sta_ID', 'Year')
    .agg(
        F.avg('Lat_Dec').alias('Lat_Dec'),
        F.avg('Lon_Dec').alias('Lon_Dec'),
        F.avg('Depthm').alias('Mean_Depthm'),
        F.avg('T_degC').alias('T_degC_Mean'),
        F.avg('Salnty').alias('Salnty_Mean'),
        F.avg('O2ml_L').alias('Current_O2_Mean'),
        F.min('O2ml_L').alias('Current_O2_Min'),
        F.avg('PO4uM').alias('PO4_Mean'),
        F.avg('NO3uM').alias('NO3_Mean'),
        F.avg('SiO3uM').alias('SiO3_Mean'),
        F.count('*').alias('Observation_Count'),
        F.countDistinct('Cst_Cnt').alias('Cast_Count')
    ))

# Preserve nulls during nonlinear projection so the training-fitted imputer handles them later.
def null_preserving_log1p(column_name):
    return F.when(F.col(column_name).isNull(), F.lit(None).cast('double')).otherwise(
        F.log1p(F.greatest(F.col(column_name), F.lit(0.0))))

def null_preserving_sqrt(column_name):
    return F.when(F.col(column_name).isNull(), F.lit(None).cast('double')).otherwise(
        F.sqrt(F.greatest(F.col(column_name), F.lit(0.0))))

# Self-join only the following year at the same station. This prevents same-year leakage.
current = station_year.alias('current')
following = station_year.select('Sta_ID', F.col('Year').alias('Target_Year'), F.col('Current_O2_Min').alias('Target_O2_Min')).alias('following')
model_data = (current.join(
    following,
    (F.col('current.Sta_ID') == F.col('following.Sta_ID')) & (F.col('current.Year') + 1 == F.col('following.Target_Year')),
    'inner')
    .select('current.*', 'following.Target_Year', 'following.Target_O2_Min')
    .withColumn('Hypoxia_Next_Year', F.when(F.col('Target_O2_Min') < 1.4, F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn('Log_PO4', null_preserving_log1p('PO4_Mean'))
    .withColumn('Log_NO3', null_preserving_log1p('NO3_Mean'))
    .withColumn('Log_SiO3', null_preserving_log1p('SiO3_Mean'))
    .withColumn('Sqrt_Depth', null_preserving_sqrt('Mean_Depthm'))
    .persist(StorageLevel.MEMORY_AND_DISK))

model_summary = model_data.agg(
    F.count('*').alias('station_year_pairs'),
    F.countDistinct('Sta_ID').alias('stations'),
    F.min('Year').alias('first_predictor_year'),
    F.max('Year').alias('last_predictor_year'),
    F.mean('Hypoxia_Next_Year').alias('next_year_warning_rate')
).toPandas()
display(model_summary)
model_summary.to_csv(OUTPUT_DIR / '05_station_year_model_summary.csv', index=False)
prepared.unpersist()


In [ ]:
# Data-quality and projection evidence for KDD Steps 2.4 and 4.2.
projection_columns = [
    'Mean_Depthm', 'Sqrt_Depth', 'PO4_Mean', 'Log_PO4',
    'NO3_Mean', 'Log_NO3', 'SiO3_Mean', 'Log_SiO3'
]
projection_pd = model_data.select(*projection_columns).toPandas()
missingness_plot = quality_report.copy()
missingness_plot['missing_pct'] = 100 - missingness_plot['completeness_pct']
missingness_plot = missingness_plot.sort_values('missing_pct', ascending=False)

fig = plt.figure(figsize=(16, 11))
grid = fig.add_gridspec(3, 4, height_ratios=[1.15, 1, 1])
missing_ax = fig.add_subplot(grid[0, :])
sns.barplot(data=missingness_plot, x='missing_pct', y='field', ax=missing_ax, color='#e15759')
missing_ax.set(title='Missing values in selected raw Bottle fields', xlabel='Missing records (%)', ylabel='')
missing_ax.set_xlim(0, 100)

projection_pairs = [
    ('Mean_Depthm', 'Sqrt_Depth', 'Mean depth (m)', 'Square-root depth'),
    ('PO4_Mean', 'Log_PO4', 'Mean phosphate (µmol/L)', 'log1p phosphate'),
    ('NO3_Mean', 'Log_NO3', 'Mean nitrate (µmol/L)', 'log1p nitrate'),
    ('SiO3_Mean', 'Log_SiO3', 'Mean silicate (µmol/L)', 'log1p silicate'),
]
for column_index, (raw_column, transformed_column, raw_label, transformed_label) in enumerate(projection_pairs):
    raw_ax = fig.add_subplot(grid[1, column_index])
    transformed_ax = fig.add_subplot(grid[2, column_index])
    sns.histplot(projection_pd[raw_column].dropna(), bins=35, ax=raw_ax, color='#4c78a8')
    sns.histplot(projection_pd[transformed_column].dropna(), bins=35, ax=transformed_ax, color='#59a14f')
    raw_ax.set(title=f'Before: {raw_label}', xlabel=raw_label, ylabel='Station-year pairs')
    transformed_ax.set(title=f'After: {transformed_label}', xlabel=transformed_label, ylabel='Station-year pairs')
fig.suptitle('Data quality and statistical projections', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_data_quality_and_transformations.png', dpi=240, bbox_inches='tight')
plt.show()


## 4. Reduction and projection

The core feature set is deliberately small: location, depth, temperature, salinity, preceding-year oxygen, and record counts. Nutrient measures are transformed with `log1p` because they are right-skewed. Depth is square-root transformed for the same reason. Missing values are explicitly preserved during projection and later imputed using medians fitted on the training period only. The extended model tests whether the nutrient features add predictive value.

In [ ]:
CORE_FEATURES = [
    'Lat_Dec', 'Lon_Dec', 'Sqrt_Depth', 'T_degC_Mean', 'Salnty_Mean',
    'Current_O2_Mean', 'Current_O2_Min', 'Observation_Count', 'Cast_Count'
]
EXTENDED_FEATURES = CORE_FEATURES + ['Log_PO4', 'Log_NO3', 'Log_SiO3']

# The time split emulates deployment: all predictor years before the last two observed years train the model.
available_years = [r['Year'] for r in model_data.select('Year').distinct().orderBy('Year').collect()]
if len(available_years) < 4:
    raise ValueError('At least four predictor years are required for a train/validation/test split.')
train_end_year = available_years[-3]
validation_year = available_years[-2]
test_year = available_years[-1]

train_df = model_data.filter(F.col('Year') <= train_end_year).cache()
validation_df = model_data.filter(F.col('Year') == validation_year).cache()
test_df = model_data.filter(F.col('Year') == test_year).cache()

split_summary = pd.DataFrame([
    {'split': 'train', 'predictor_years': f'{available_years[0]}–{train_end_year}', 'station_year_pairs': train_df.count()},
    {'split': 'validation', 'predictor_years': str(validation_year), 'station_year_pairs': validation_df.count()},
    {'split': 'test', 'predictor_years': str(test_year), 'station_year_pairs': test_df.count()},
])
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / '06_temporal_split.csv', index=False)
model_data.unpersist()


## 5–7. Modelling, test design and data mining

The regression task predicts next-year minimum oxygen. The classification task produces a warning where next-year minimum oxygen is below 1.4 mL/L. Linear/logistic regression provide transparent baselines; random forests test nonlinear relationships. Algorithm, feature-set and random-forest parameter selection use the validation year only. The final test year remains unseen until every selection decision is locked.

In [ ]:
def prepare_feature_frames(train_source, evaluation_source, feature_columns):
    # Fit imputation only on the training period, then reuse it for every candidate model.
    imputed = [f'{c}__imputed' for c in feature_columns]
    preprocessor = Pipeline(stages=[
        Imputer(inputCols=feature_columns, outputCols=imputed, strategy='median'),
        VectorAssembler(inputCols=imputed, outputCol='features', handleInvalid='skip'),
    ]).fit(train_source)
    # Retain identifiers and coordinates for the later station-level output map.
    prepared_train = preprocessor.transform(train_source).persist(StorageLevel.MEMORY_AND_DISK)
    prepared_evaluation = preprocessor.transform(evaluation_source).persist(StorageLevel.MEMORY_AND_DISK)
    # Materialise once, so later candidate fits do not repeat the full Spark lineage.
    prepared_train.count()
    prepared_evaluation.count()
    return prepared_train, prepared_evaluation

def regression_metrics(predictions):
    return {
        'RMSE': RegressionEvaluator(labelCol='Target_O2_Min', predictionCol='prediction', metricName='rmse').evaluate(predictions),
        'MAE': RegressionEvaluator(labelCol='Target_O2_Min', predictionCol='prediction', metricName='mae').evaluate(predictions),
        'R2': RegressionEvaluator(labelCol='Target_O2_Min', predictionCol='prediction', metricName='r2').evaluate(predictions),
    }

def classification_metrics(predictions):
    return {
        'ROC_AUC': BinaryClassificationEvaluator(labelCol='Hypoxia_Next_Year', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions),
        'F1': MulticlassClassificationEvaluator(labelCol='Hypoxia_Next_Year', predictionCol='prediction', metricName='f1').evaluate(predictions),
        'Accuracy': MulticlassClassificationEvaluator(labelCol='Hypoxia_Next_Year', predictionCol='prediction', metricName='accuracy').evaluate(predictions),
    }

def classification_point_metrics(predictions):
    # Explicit confusion counts make the warning errors auditable and also support simple baselines.
    counts = predictions.agg(
        F.sum(F.when((F.col('Hypoxia_Next_Year') == 1) & (F.col('prediction') == 1), 1).otherwise(0)).alias('TP'),
        F.sum(F.when((F.col('Hypoxia_Next_Year') == 0) & (F.col('prediction') == 0), 1).otherwise(0)).alias('TN'),
        F.sum(F.when((F.col('Hypoxia_Next_Year') == 0) & (F.col('prediction') == 1), 1).otherwise(0)).alias('FP'),
        F.sum(F.when((F.col('Hypoxia_Next_Year') == 1) & (F.col('prediction') == 0), 1).otherwise(0)).alias('FN'),
    ).first().asDict()
    tp, tn, fp, fn = [int(counts[key]) for key in ['TP', 'TN', 'FP', 'FN']]
    total = tp + tn + fp + fn
    safe_ratio = lambda numerator, denominator: numerator / denominator if denominator else 0.0
    precision_1 = safe_ratio(tp, tp + fp)
    recall_1 = safe_ratio(tp, tp + fn)
    precision_0 = safe_ratio(tn, tn + fn)
    recall_0 = safe_ratio(tn, tn + fp)
    f1_1 = safe_ratio(2 * precision_1 * recall_1, precision_1 + recall_1)
    f1_0 = safe_ratio(2 * precision_0 * recall_0, precision_0 + recall_0)
    weighted_f1 = safe_ratio((tp + fn) * f1_1 + (tn + fp) * f1_0, total)
    return {
        'F1': weighted_f1, 'Accuracy': safe_ratio(tp + tn, total),
        'Precision': precision_1, 'Recall': recall_1, 'Specificity': recall_0,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
    }

# Exploratory models are evaluated on the validation year only.
# Preprocessing is shared across candidates to avoid repeated Spark jobs.
train_core, validation_core = prepare_feature_frames(train_df, validation_df, CORE_FEATURES)

regression_candidates = {
    'Linear regression': LinearRegression(maxIter=100, regParam=0.05, elasticNetParam=0.1),
    'Random forest regression': RandomForestRegressor(numTrees=80, maxDepth=6, seed=722),
}
classification_candidates = {
    'Logistic regression': LogisticRegression(maxIter=100, regParam=0.05),
    'Random forest classifier': RandomForestClassifier(numTrees=80, maxDepth=6, seed=722),
}

exploration_rows = []
for name, estimator in regression_candidates.items():
    fitted = estimator.setLabelCol('Target_O2_Min').setFeaturesCol('features').setPredictionCol('prediction').fit(train_core)
    predictions = fitted.transform(validation_core).persist(StorageLevel.MEMORY_AND_DISK)
    predictions.count()
    metrics = regression_metrics(predictions)
    exploration_rows.append({'task': 'regression', 'model': name, **metrics})
    predictions.unpersist()

for name, estimator in classification_candidates.items():
    fitted = estimator.setLabelCol('Hypoxia_Next_Year').setFeaturesCol('features').setPredictionCol('prediction').fit(train_core)
    predictions = fitted.transform(validation_core).persist(StorageLevel.MEMORY_AND_DISK)
    predictions.count()
    metrics = classification_metrics(predictions)
    exploration_rows.append({'task': 'classification', 'model': name, **metrics})
    predictions.unpersist()

exploration_results = pd.DataFrame(exploration_rows)
display(exploration_results)
exploration_results.to_csv(OUTPUT_DIR / '07_validation_model_comparison.csv', index=False)

# Feature-set iteration also uses the validation year. The held-out test year is not inspected here.
train_extended, validation_extended = prepare_feature_frames(train_df, validation_df, EXTENDED_FEATURES)
extended_validation_regression = (RandomForestRegressor(numTrees=80, maxDepth=6, seed=722)
    .setLabelCol('Target_O2_Min').setFeaturesCol('features').setPredictionCol('prediction')
    .fit(train_extended).transform(validation_extended).persist(StorageLevel.MEMORY_AND_DISK))
extended_validation_classifier = (RandomForestClassifier(numTrees=80, maxDepth=6, seed=722)
    .setLabelCol('Hypoxia_Next_Year').setFeaturesCol('features').setPredictionCol('prediction')
    .fit(train_extended).transform(validation_extended).persist(StorageLevel.MEMORY_AND_DISK))
extended_validation_regression.count()
extended_validation_classifier.count()
core_rf_regression = next(row for row in exploration_rows if row['model'] == 'Random forest regression')
core_rf_classifier = next(row for row in exploration_rows if row['model'] == 'Random forest classifier')
iteration_results = pd.DataFrame([
    {'evaluation_period': f'validation predictor year {validation_year}', 'feature_set': 'Core', **core_rf_regression},
    {'evaluation_period': f'validation predictor year {validation_year}', 'feature_set': 'Extended nutrients', 'task': 'regression', 'model': 'Random forest regression', **regression_metrics(extended_validation_regression)},
    {'evaluation_period': f'validation predictor year {validation_year}', 'feature_set': 'Core', **core_rf_classifier},
    {'evaluation_period': f'validation predictor year {validation_year}', 'feature_set': 'Extended nutrients', 'task': 'classification', 'model': 'Random forest classifier', **classification_metrics(extended_validation_classifier)},
])
display(iteration_results)
iteration_results.to_csv(OUTPUT_DIR / '13_iteration_core_vs_extended.csv', index=False)
train_extended.unpersist()
validation_extended.unpersist()
extended_validation_regression.unpersist()
extended_validation_classifier.unpersist()

# Small validation-only parameter comparison for rubric criterion 6.3.
# The compact setting reuses the exploratory result; only the stability-focused setting requires two additional fits.
stability_validation_regression = (RandomForestRegressor(numTrees=250, maxDepth=8, minInstancesPerNode=3, seed=722)
    .setLabelCol('Target_O2_Min').setFeaturesCol('features').setPredictionCol('prediction')
    .fit(train_core).transform(validation_core).persist(StorageLevel.MEMORY_AND_DISK))
stability_validation_classifier = (RandomForestClassifier(numTrees=250, maxDepth=8, minInstancesPerNode=3, seed=722)
    .setLabelCol('Hypoxia_Next_Year').setFeaturesCol('features').setPredictionCol('prediction')
    .fit(train_core).transform(validation_core).persist(StorageLevel.MEMORY_AND_DISK))
stability_validation_regression.count()
stability_validation_classifier.count()
parameter_rows = [
    {'task': 'regression', 'configuration': 'Compact', 'num_trees': 80, 'max_depth': 6, 'min_instances_per_node': 1, 'seed': 722,
     'RMSE': core_rf_regression['RMSE'], 'MAE': core_rf_regression['MAE'], 'R2': core_rf_regression['R2']},
    {'task': 'regression', 'configuration': 'Stability-focused', 'num_trees': 250, 'max_depth': 8, 'min_instances_per_node': 3, 'seed': 722,
     **regression_metrics(stability_validation_regression)},
    {'task': 'classification', 'configuration': 'Compact', 'num_trees': 80, 'max_depth': 6, 'min_instances_per_node': 1, 'seed': 722,
     'ROC_AUC': core_rf_classifier['ROC_AUC'], 'F1': core_rf_classifier['F1'], 'Accuracy': core_rf_classifier['Accuracy']},
    {'task': 'classification', 'configuration': 'Stability-focused', 'num_trees': 250, 'max_depth': 8, 'min_instances_per_node': 3, 'seed': 722,
     **classification_metrics(stability_validation_classifier)},
]
regression_parameter_candidates = [row for row in parameter_rows if row['task'] == 'regression']
classification_parameter_candidates = [row for row in parameter_rows if row['task'] == 'classification']
selected_regression_parameters = min(
    regression_parameter_candidates,
    key=lambda row: (row['RMSE'], row['MAE'], row['num_trees'], row['max_depth']))
selected_classification_parameters = max(
    classification_parameter_candidates,
    key=lambda row: (row['ROC_AUC'], row['F1'], row['Accuracy'], -row['num_trees'], -row['max_depth']))
for row in parameter_rows:
    selected_row = selected_regression_parameters if row['task'] == 'regression' else selected_classification_parameters
    row['selected'] = row['configuration'] == selected_row['configuration']
parameter_comparison = pd.DataFrame(parameter_rows)
display(parameter_comparison)
parameter_comparison.to_csv(OUTPUT_DIR / '17_validation_parameter_comparison.csv', index=False)
SELECTED_RF_PARAMETERS = {
    'regression': {
        'numTrees': int(selected_regression_parameters['num_trees']),
        'maxDepth': int(selected_regression_parameters['max_depth']),
        'minInstancesPerNode': int(selected_regression_parameters['min_instances_per_node']),
        'seed': int(selected_regression_parameters['seed']),
    },
    'classification': {
        'numTrees': int(selected_classification_parameters['num_trees']),
        'maxDepth': int(selected_classification_parameters['max_depth']),
        'minInstancesPerNode': int(selected_classification_parameters['min_instances_per_node']),
        'seed': int(selected_classification_parameters['seed']),
    },
}
SELECTED_RF_VALIDATION_METRICS = {
    'regression': selected_regression_parameters,
    'classification': selected_classification_parameters,
}
stability_validation_regression.unpersist()
stability_validation_classifier.unpersist()
train_core.unpersist()
validation_core.unpersist()


In [ ]:
# Validation-only visual evidence for algorithm, feature-set and parameter decisions.
algorithm_regression = exploration_results[exploration_results['task'] == 'regression'].copy()
algorithm_classification = exploration_results[exploration_results['task'] == 'classification'].copy()
feature_regression = iteration_results[iteration_results['task'] == 'regression'].copy()
parameter_regression = parameter_comparison[parameter_comparison['task'] == 'regression'].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.barplot(data=algorithm_regression, x='model', y='RMSE', ax=axes[0, 0], color='#4c78a8')
axes[0, 0].set(title='Regression algorithm comparison', xlabel='', ylabel='Validation RMSE (mL/L)')
axes[0, 0].tick_params(axis='x', rotation=15)
sns.barplot(data=algorithm_classification, x='model', y='Accuracy', ax=axes[0, 1], color='#f28e2b')
axes[0, 1].set(title='Classification algorithm comparison', xlabel='', ylabel='Validation accuracy', ylim=(0, 1.05))
axes[0, 1].tick_params(axis='x', rotation=15)
sns.barplot(data=feature_regression, x='feature_set', y='RMSE', ax=axes[1, 0], color='#59a14f')
axes[1, 0].set(title='Regression feature-set iteration', xlabel='', ylabel='Validation RMSE (mL/L)')
sns.barplot(data=parameter_regression, x='configuration', y='RMSE', ax=axes[1, 1], color='#e15759')
axes[1, 1].set(title='Regression parameter comparison', xlabel='', ylabel='Validation RMSE (mL/L)')
for ax in axes.flat:
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
fig.suptitle(f'Validation-only model selection: predictor year {validation_year}', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07b_validation_model_selection.png', dpi=240, bbox_inches='tight')
plt.show()


In [ ]:
# Select the random forests after exploratory comparison, retrain using train + validation data, then evaluate once on the held-out final year.
from time import perf_counter
started_at = perf_counter()

def show_elapsed(stage):
    print(f'[{perf_counter() - started_at:,.1f}s] {stage}', flush=True)

show_elapsed('Caching training and validation records')
train_validation_df = train_df.unionByName(validation_df).persist(StorageLevel.MEMORY_AND_DISK)
train_validation_df.count()
train_df.unpersist()
validation_df.unpersist()

# Fit preprocessing once on train + validation, then reuse the prepared frames in both final models.
show_elapsed('Preparing core features')
train_validation_core, test_core = prepare_feature_frames(train_validation_df, test_df, CORE_FEATURES)
regression_parameters = SELECTED_RF_PARAMETERS['regression']
classification_parameters = SELECTED_RF_PARAMETERS['classification']
show_elapsed(f"Fitting final random-forest regression ({regression_parameters['numTrees']} trees)")
final_regression = (RandomForestRegressor(**regression_parameters)
    .setLabelCol('Target_O2_Min').setFeaturesCol('features').setPredictionCol('prediction')
    .fit(train_validation_core))
show_elapsed(f"Fitting final random-forest classifier ({classification_parameters['numTrees']} trees)")
final_classifier = (RandomForestClassifier(**classification_parameters)
    .setLabelCol('Hypoxia_Next_Year').setFeaturesCol('features').setPredictionCol('prediction')
    .fit(train_validation_core))
selected_regression_validation = SELECTED_RF_VALIDATION_METRICS['regression']
selected_classification_validation = SELECTED_RF_VALIDATION_METRICS['classification']
parameter_evidence = pd.DataFrame([
    {'task': 'regression', 'algorithm': 'RandomForestRegressor', 'feature_set': 'Core',
     'configuration': selected_regression_validation['configuration'],
     'num_trees': regression_parameters['numTrees'], 'max_depth': regression_parameters['maxDepth'],
     'min_instances_per_node': regression_parameters['minInstancesPerNode'], 'seed': regression_parameters['seed'],
     'validation_RMSE': selected_regression_validation['RMSE'], 'validation_MAE': selected_regression_validation['MAE'],
     'selection_basis': 'Lowest validation RMSE; MAE and lower complexity break ties.'},
    {'task': 'classification', 'algorithm': 'RandomForestClassifier', 'feature_set': 'Core',
     'configuration': selected_classification_validation['configuration'],
     'num_trees': classification_parameters['numTrees'], 'max_depth': classification_parameters['maxDepth'],
     'min_instances_per_node': classification_parameters['minInstancesPerNode'], 'seed': classification_parameters['seed'],
     'validation_ROC_AUC': selected_classification_validation['ROC_AUC'], 'validation_F1': selected_classification_validation['F1'],
     'selection_basis': 'Highest validation ROC-AUC, then weighted F1 and accuracy; lower complexity breaks ties.'},
])
display(parameter_evidence)
parameter_evidence.to_csv(OUTPUT_DIR / '15_final_model_parameters.csv', index=False)

# Materialise predictions once: all metrics below then reuse cached outputs.
show_elapsed('Generating and caching held-out predictions')
test_regression = final_regression.transform(test_core).persist(StorageLevel.MEMORY_AND_DISK)
test_classification = final_classifier.transform(test_core).persist(StorageLevel.MEMORY_AND_DISK)
test_regression.count()
test_classification.count()
train_validation_core.unpersist()
test_core.unpersist()

show_elapsed('Calculating held-out metrics')
# Compare the selected models with transparent no-learning baselines on exactly the same held-out rows.
persistence_regression = test_regression.withColumn('prediction', F.col('Current_O2_Min'))
persistence_classifier = test_classification.withColumn(
    'prediction', F.when(F.col('Current_O2_Min') < 1.4, F.lit(1.0)).otherwise(F.lit(0.0)))
current_oxygen_ranking = test_classification.withColumn('Current_O2_Risk_Score', -F.col('Current_O2_Min'))
current_oxygen_ranking_auc = BinaryClassificationEvaluator(
    labelCol='Hypoxia_Next_Year', rawPredictionCol='Current_O2_Risk_Score', metricName='areaUnderROC'
).evaluate(current_oxygen_ranking)
always_warning_classifier = test_classification.withColumn('prediction', F.lit(1.0))
rf_classification_metrics = {**classification_metrics(test_classification), **classification_point_metrics(test_classification)}
test_metrics = pd.DataFrame([
    {'task': 'regression', 'model': 'Current-year persistence baseline', **regression_metrics(persistence_regression)},
    {'task': 'regression', 'model': 'Random forest', **regression_metrics(test_regression)},
    {'task': 'classification', 'model': 'Always-warning baseline', **classification_point_metrics(always_warning_classifier)},
    {'task': 'classification', 'model': 'Current-O2 persistence and ranking baseline',
     'ROC_AUC': current_oxygen_ranking_auc, **classification_point_metrics(persistence_classifier)},
    {'task': 'classification', 'model': 'Random forest', **rf_classification_metrics},
])
display(test_metrics)
test_metrics.to_csv(OUTPUT_DIR / '08_final_test_metrics.csv', index=False)
show_elapsed('Final-model evaluation complete')


## 8. Evaluate patterns, visualise results and iterate

The held-out evidence includes transparent persistence baselines, a continuous current-oxygen ranking baseline, explicit confusion counts, an observed-versus-predicted regression plot, a classification confusion matrix and a station warning map. The map displays the random-forest warning score: squares mark model warnings at score ≥ 0.50 and circles mark observed following-year warning outcomes. The score is not treated as a calibrated probability. Overlap is a correct warning. Feature importance is descriptive rather than causal.

In [ ]:
# Combine the continuous estimate and warning score in one auditable station-level evaluation table.
regression_output = test_regression.select(
    'Sta_ID', 'Year', 'Target_Year', F.col('prediction').alias('Predicted_O2_Min'))
prediction_output = (test_classification
    .select('Sta_ID', 'Year', 'Target_Year', 'Lat_Dec', 'Lon_Dec', 'Current_O2_Mean', 'Current_O2_Min', 'Target_O2_Min',
            'Hypoxia_Next_Year', 'probability', 'prediction')
    .withColumn('Warning_Score', vector_to_array('probability')[1])
    .withColumn('Model_Warning', F.when(F.col('Warning_Score') >= 0.50, 1).otherwise(0))
    .drop('probability', 'prediction')
    .join(regression_output, ['Sta_ID', 'Year', 'Target_Year'], 'left')
    .select('Sta_ID', 'Year', 'Target_Year', 'Lat_Dec', 'Lon_Dec', 'Current_O2_Mean', 'Current_O2_Min',
            'Predicted_O2_Min', 'Target_O2_Min', 'Hypoxia_Next_Year', 'Warning_Score', 'Model_Warning'))
if prediction_output.filter(F.col('Predicted_O2_Min').isNull()).limit(1).count():
    raise ValueError('Regression predictions did not align with classification predictions.')
prediction_pd = prediction_output.toPandas()
prediction_pd.to_csv(OUTPUT_DIR / '09_held_out_station_predictions.csv', index=False)
display(prediction_pd.sort_values('Warning_Score', ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(prediction_pd['Lon_Dec'], prediction_pd['Lat_Dec'], c=prediction_pd['Warning_Score'],
                     cmap='YlOrRd', s=55, alpha=0.9, label='All held-out stations')
warned = prediction_pd[prediction_pd['Model_Warning'] == 1]
observed = prediction_pd[prediction_pd['Hypoxia_Next_Year'] == 1]
ax.scatter(warned['Lon_Dec'], warned['Lat_Dec'], facecolors='none', edgecolors='purple', marker='s', s=180, linewidths=1.8, label='Model warning')
ax.scatter(observed['Lon_Dec'], observed['Lat_Dec'], facecolors='none', edgecolors='black', s=180, linewidths=1.8, label='Observed next-year warning')
plt.colorbar(scatter, ax=ax, label='Random-forest warning score')
ax.set(title=f'Retrospective one-year-ahead warnings: predictor year {test_year}', xlabel='Longitude', ylabel='Latitude')
ax.legend(loc='best')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '10_held_out_warning_map.png', dpi=240, bbox_inches='tight')
plt.show()

# Diagnostic evidence complements aggregate metrics by showing error structure on the held-out year.
regression_pd = prediction_pd[['Target_O2_Min', 'Predicted_O2_Min']].copy()
confusion = pd.crosstab(
    prediction_pd['Hypoxia_Next_Year'].astype(int),
    prediction_pd['Model_Warning'].astype(int),
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(regression_pd['Target_O2_Min'], regression_pd['Predicted_O2_Min'], alpha=0.75, color='#1f5a99')
axis_min = min(regression_pd['Target_O2_Min'].min(), regression_pd['Predicted_O2_Min'].min())
axis_max = max(regression_pd['Target_O2_Min'].max(), regression_pd['Predicted_O2_Min'].max())
axes[0].plot([axis_min, axis_max], [axis_min, axis_max], linestyle='--', color='black', linewidth=1)
axes[0].set(title='Held-out regression: observed vs predicted', xlabel='Observed next-year minimum O2 (mL/L)', ylabel='Predicted next-year minimum O2 (mL/L)')
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1])
axes[1].set(title='Held-out warning confusion matrix', xlabel='Model warning', ylabel='Observed warning')
axes[1].set_xticklabels(['No', 'Yes'])
axes[1].set_yticklabels(['No', 'Yes'], rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '16_evaluation_diagnostics.png', dpi=240, bbox_inches='tight')
plt.show()


In [ ]:
# Built-in random-forest feature importance from the selected models.
reg_rf = final_regression
clf_rf = final_classifier
importance = pd.DataFrame({
    'feature': CORE_FEATURES,
    'regression_importance': reg_rf.featureImportances.toArray(),
    'classification_importance': clf_rf.featureImportances.toArray(),
}).sort_values('classification_importance', ascending=False)
importance.to_csv(OUTPUT_DIR / '11_feature_importance.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=importance.sort_values('regression_importance'), x='regression_importance', y='feature', ax=axes[0], color='#4c78a8')
axes[0].set(title='Regression feature importance', xlabel='Importance', ylabel='')
sns.barplot(data=importance.sort_values('classification_importance'), x='classification_importance', y='feature', ax=axes[1], color='#e15759')
axes[1].set(title='Classification feature importance', xlabel='Importance', ylabel='')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '12_feature_importance.png', dpi=240, bbox_inches='tight')
plt.show()


In [ ]:
# Feature-set selection was completed on the validation year above. No further model choice uses the held-out year.
print('Evidence files created:', len(list(OUTPUT_DIR.iterdir())))
train_validation_df.unpersist()
test_df.unpersist()
test_regression.unpersist()
test_classification.unpersist()


## 9. Export submission evidence

The final cell packages the reproducible evidence: CSV tables and PNG figures produced by the notebook. Download the notebook itself from Colab (`File → Download → Download .ipynb`) and include it in the same submission zip. Do not claim a live operational warning: these are retrospective model outputs and need local threshold governance, expert review and future prospective validation.

In [ ]:
# Package outputs. Add the downloaded .ipynb to this archive before Canvas submission.
archive_path = shutil.make_archive('/content/A5_CalCOFI_BDAS_outputs', 'zip', OUTPUT_DIR)
print('Created:', archive_path)
from google.colab import files
files.download(archive_path)
